#### This notebook investigates whether we can significantly reduce our search space for the target retrieval task!
Many retrieval-based codes that other participants share are based on (relatively) small text encoders that might have lower performance when encoding articles. Moreover, all the questions are based on STEM topics, so do we really need to retrieve over the whole Wikipedia, **OR** there exists some cluster of articles that include the majority of needed contexts?
This notebook addresses both issues by using embeddings that are generated by a powerful text encoder from the [Cohere](https://txt.cohere.com/embedding-archives-wikipedia) team, and also showing that there are useful clusters that contain 95%+ of the train dataset's relevant articles. It's worth mentioning that as the text encoder is only accessible through API, we can't directly use these embeddings for end-to-end retrieval.

The final set of filtered articles is also released. (see below!)

In [ ]:
import datasets
from datasets import load_dataset
from tqdm import tqdm
import collections
import os
import numpy as np
import torch
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

#### First let's get the relevant articles for the questions in the train dataset. ([source](https://www.kaggle.com/competitions/kaggle-llm-science-exam/discussion/425242))

In [ ]:
target_articles = ['API gravity', 'Amplitude', 'Angular momentum', 'Antiferromagnetism', 'Astrochemistry', 'Baryogenesis', 'Black hole', 'Bollard pull', 'Born reciprocity', 'Butterfly effect', 'C1 chemistry', 'Causality (physics)', 'Cavitation', 'Clockwise', 'Coffee ring effect', 'Coherence (physics)', 'Coherent turbulent structure', 'Cold dark matter', "Commentary on Anatomy in Avicenna's Canon", 'Condensation cloud', 'Convection (heat transfer)', 'Copernican principle', 'Critical Raw Materials Act', 'Crossover experiment (chemistry)', 'Crystallinity', 'Dark Matter', 'Decay technique', 'Diffraction', 'Dimension', 'Droste effect', 'Dynamic scaling', "Earnshaw's theorem", 'Ecological pyramid', 'Electric flux', 'Electrical resistivity and conductivity', 'Electrochemical gradient', 'Electronic entropy', "Elitzur's theorem", 'Emissivity', 'Enthalpy', 'Environmental Science Center', 'Erlangen program', 'Explicit symmetry breaking', "Fermat's principle", 'Ferromagnetism', 'Frame-dragging', 'Free neutron decay', 'Galaxy', 'Geometric quantization', 'Gravitational wave', 'Gravity Probe B', 'Heart', 'Heat treating', "Hesse's principle of transfer", 'History of geology', 'Hydrodynamic stability', 'Improper rotation', 'Infectious tolerance', 'Inflation (cosmology)', 'Interstellar medium', 'James Webb Space Telescope', 'Kutta-Joukowski theorem', 'Landau–Lifshitz–Gilbert equation', 'Leidenfrost effect', 'Light-year', 'Linear time-invariant system', 'List of equations in classical mechanics', 'Lorentz covariance', 'Luminance', 'Magnetic monopole', 'Magnetic resonance imaging', 'Magnetic susceptibility', 'Magnitude (astronomy)', 'Main sequence', 'Mammary gland', 'Mass versus weight', 'Mass-to-charge ratio', 'Memristor', 'Minkowski space', 'Modified Newtonian dynamics', 'Molecular cloud', 'Molecular symmetry', 'Morphology (biology)', 'Navier–Stokes equations', 'Nebula', "Newton's law of universal gravitation", 'Nuclear fusion', 'Observable universe', 'Organography', 'Paramagnetism', 'Parity (physics)', 'Phageome', 'Phase transition', 'Photophoresis', 'Planetary system', 'Plant', 'Point groups in three dimensions', 'Probability amplitude', 'Probability density function', 'Propagation constant', 'Pulsar', 'Pulsar-based navigation', 'QCD matter', 'Quantization (physics)', 'Quartz crystal microbalance', 'Radiosity (radiometry)', 'Ramsauer–Townsend effect', 'Rayleigh scattering', 'Reciprocal length', 'Redshift', 'Refractive index', 'Regular polytope', 'Relative density', 'Renormalization', 'Ring-imaging Cherenkov detector', 'Scale (ratio)', 'Second law of thermodynamics', 'Self-organization in cybernetics', 'Shower-curtain effect', 'Signal-to-noise ratio', 'Spatial dispersion', 'Speed of light', 'Spin (physics)', 'Spontaneous symmetry breaking', 'Standard Model', 'Stellar classification', 'Stochastic differential equation', 'Superconductivity', 'Supermassive black hole', 'Supernova', 'Supersymmetric quantum mechanics', 'Supersymmetry', 'Surface power density', 'Surgical pathology', 'Symmetry in biology', 'Symmetry of diatomic molecules', 'The Ambidextrous Universe', 'Theorem', 'Theorem of three moments', 'Thermal equilibrium', 'Tidal force', 'Time', 'Time standard', 'Total internal reflection', 'Triskelion', 'Ultraviolet catastrophe', 'Unified field theory', 'Uniform tilings in hyperbolic plane', 'Vacuum', 'Virtual particle', 'Water hammer', 'Wigner quasiprobability distribution', 'Work function', 'Zero-point energy']

### Load Cohere embeddings for wikipedia

In [ ]:
cohere_dataset = load_dataset(f"Cohere/wikipedia-22-12-en-embeddings", split="train")

In [ ]:
type(cohere_dataset)

#### keeping only the first paragraph for each article for the clustering

In [ ]:
first_paraphs = cohere_dataset.filter(lambda example: example["paragraph_id"]==0, num_proc=20) #5.7 million

In [ ]:
cohere_titles = cohere_dataset["title"]
cohere_first_paraph_titles = first_paraphs["title"]

docs_titles = first_paraphs["title"]
docs_embeddings = first_paraphs["emb"]

In [ ]:
doc_embeddings_array = np.array(docs_embeddings)
# del docs_embeddings

#### doing Kmeans over the embeddings of the first paragpraph of all articles

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

kmean_dict = {}
for num_clusters in [20,]:
    kmean_dict[num_clusters] = KMeans(n_clusters=num_clusters,
                                      random_state=0,
                                      n_init=1).fit(doc_embeddings_array)

In [ ]:
cluster_numbers = kmean_dict[20].predict(doc_embeddings_array)

cluster_dict = collections.defaultdict(list) # membership dictionary for each cluster
for dx,cl_id in enumerate(cluster_numbers):
    cluster_dict[cl_id].append(cohere_first_paraph_titles[dx])

#### checking cluster membership of all the target articles

In [ ]:
# only two articles not in the majority cluster
sorted([(cluster_numbers[dx],i) for dx,i in enumerate(cohere_first_paraph_titles) if i in target_articles])

### choosing cluster:13 for the following filtering as it has only two articles missing from the train set

In [ ]:
filtered_articles = cluster_dict[13] #363142 articles
filtered_articles_set = set(filtered_articles)

In [ ]:
first_paraphs_filtered = first_paraphs.filter(lambda example: example["title"] in filtered_articles_set,
                                              num_proc=20)

In [ ]:
docs_titles_filtered = first_paraphs_filtered["title"]
docs_embeddings_filtered = first_paraphs_filtered["emb"]

In [ ]:
doc_embeddings_array_filtered = np.array(docs_embeddings_filtered)

In [ ]:
kmean_dict_filtered = {}
for num_clusters in tqdm([6]):
    kmean_dict_filtered[num_clusters] = KMeans(n_clusters=num_clusters,
                                               random_state=0,
                                               n_init=1).fit(doc_embeddings_array_filtered)

In [ ]:
cluster_numbers_filtered = kmean_dict_filtered[6].predict(doc_embeddings_array_filtered)

cluster_dict = collections.defaultdict(list)
for dx,cl_id in enumerate(cluster_numbers_filtered):
    cluster_dict[cl_id].append(docs_titles_filtered[dx])

#### checking cluster membership of all the target articles

In [ ]:
sorted([(cluster_numbers_filtered[dx],i) for dx,i in enumerate(docs_titles_filtered) if i in target_articles])

### filtering out 4th cluster as it's the largest and also irrelevant. keeping the 5th cluster as the test set could shift

In [ ]:
filtered_articles1 = cluster_dict[0] + cluster_dict[1] + cluster_dict[2] + cluster_dict[4] + cluster_dict[5] #277046 articles
filtered_articles_set1 = set(filtered_articles1)

In [ ]:
first_paraphs_filtered1 = first_paraphs_filtered.filter(lambda example: example["title"] in filtered_articles_set1,
                                                        num_proc=20)

In [ ]:
docs_titles_filtered1 = first_paraphs_filtered1["title"]
docs_embeddings_filtered1 = first_paraphs_filtered1["emb"]

In [ ]:
cohere_dataset_filtered = cohere_dataset.filter(lambda example: example["title"] in filtered_articles_set1,
                                                num_proc=20)

In [ ]:
# saving the filtered articles (without embeddings) to disk
cohere_dataset_filtered.remove_columns(["emb"]).save_to_disk("cohere_dataset_filtered_noEmb.hf")

### The filtered set of articles can be found [here](https://www.kaggle.com/datasets/mbanaei/stem-wiki-cohere-no-emb)!